# DISSCO — Listen While It Renders

Renders a `.dissco` piece with the deterministic pipeline and plays the audio
**while it is still rendering** (restructure/10_REALTIME_LISTENING.md).
On this stack a 10-minute piece streams at ~30× real-time, so playback starts
seconds after launch and never starves.

- The stream is a **preview** (pre-anticlip, clamped); the authoritative AIFF
  written at the end is byte-identical to a normal render.
- Browser autoplay policies may require one click on the first player widget.
- Requires: a built `cmod` (repo root), `numpy`, and a browser for audio.

In [ ]:
# ---- configuration ----------------------------------------------------
import os, sys, pathlib
REPO   = pathlib.Path.cwd()
while not (REPO / 'cmod').exists() and REPO != REPO.parent: REPO = REPO.parent
CMOD   = REPO / 'cmod'
PIECE  = REPO / 'Tutorial.dissco'      # <- point at your piece
SEED   = 42
THREADS= 20
STREAM = pathlib.Path('/tmp/dissco_live.pcm')
ENV = dict(os.environ,
           LASS_COMPOSITE='det',           # required for streaming
           LASS_PIPELINE='gpu-fast',       # optional: fused GPU pipeline
           LASS_STREAM=str(STREAM))
sys.path.insert(0, str(REPO / 'restructure' / 'realtime'))
from listen import wait_for_header, tail_windows
print(f'cmod: {CMOD}\npiece: {PIECE}')

In [ ]:
# ---- launch + listen ----------------------------------------------------
import subprocess, tempfile, time, re
import numpy as np
from IPython.display import Audio, display

# stage the piece with our seed/threads in a scratch dir (originals untouched)
work = pathlib.Path(tempfile.mkdtemp(prefix='dissco_live_'))
txt = PIECE.read_text()
txt = re.sub(r'<Seed>[^<]*</Seed>', f'<Seed>{SEED}</Seed>', txt)
txt = re.sub(r'<NumberOfThreads>\\d+</NumberOfThreads>',
             f'<NumberOfThreads>{THREADS}</NumberOfThreads>', txt)
(work / 'live.dissco').write_text(txt)
STREAM.unlink(missing_ok=True)

t0 = time.time()
proc = subprocess.Popen(f'echo 1 | "{CMOD}" live.dissco', shell=True, cwd=work,
                        env=ENV, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
rate, ch = wait_for_header(str(STREAM))
print(f'stream up: {rate} Hz, {ch} ch')

CHUNK_S = 2.0                      # seconds of audio per player widget
buf = b''
first = None
played_s = 0.0
for t_wall, data in tail_windows(str(STREAM), rate, ch, idle_timeout=20.0):
    if first is None:
        first = t_wall
        print(f'time-to-first-audio: {first - t0:.2f}s')
    buf += data
    frame = 4 * ch
    while len(buf) >= int(CHUNK_S * rate) * frame:
        cut = int(CHUNK_S * rate) * frame
        chunk, buf = buf[:cut], buf[cut:]
        pcm = np.frombuffer(chunk, dtype='<f4').reshape(-1, ch)
        display(Audio(pcm.T, rate=rate, autoplay=True, normalize=False))
        played_s += CHUNK_S
        # pace: stay just behind the render frontier so widgets play seamlessly
        ahead = played_s - (time.time() - first)
        if ahead > CHUNK_S: time.sleep(ahead - CHUNK_S * 0.5)
if buf:
    pcm = np.frombuffer(buf, dtype='<f4').reshape(-1, ch)
    display(Audio(pcm.T, rate=rate, autoplay=True, normalize=False))
proc.wait()
print(f'render complete; authoritative AIFF in {work}/SoundFiles/')